In [1]:
# Run this in a cell before exporting
%pip install onnx onnxsim onnxruntime-gpu

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
from ultralytics import YOLO
import yaml
import torch
torch.cuda.empty_cache()


PEST_DATASET_PATH = "rebalanced_dataset"
RACCOON_DATASET_PATH = "rebalanced_raccoon_dataset"

In [3]:
def create_dataset_yaml(dataset_path, num_classes=1, class_names={0: 'raccoon'}, output_yaml="data.yaml"):
    """Create dataset YAML configuration with proper formatting"""
    
    yaml_content = {
        'path': os.path.abspath(dataset_path),
        'train': 'train/images',
        'val': 'valid/images', 
        'test': 'test/images',
        'nc': num_classes,
        'names': class_names
    }
    
    # Save YAML
    with open(output_yaml, "w") as f:
        yaml.dump(yaml_content, f, default_flow_style=False, sort_keys=False)
    
    print(f"{output_yaml} created!")
    print(f"Dataset path: {os.path.abspath(dataset_path)}")
    return output_yaml

# Create dataset configuration
yaml_path = create_dataset_yaml(PEST_DATASET_PATH, num_classes=2, class_names={0: 'raccoon', 1: 'rodent'}, output_yaml="pest.yaml") 
yaml_path = create_dataset_yaml(RACCOON_DATASET_PATH, num_classes=1, class_names={0: 'raccoon'}, output_yaml="raccoon.yaml") 

pest.yaml created!
Dataset path: c:\Users\plebj\Desktop\School\CS5814\rebalanced_dataset
raccoon.yaml created!
Dataset path: c:\Users\plebj\Desktop\School\CS5814\rebalanced_raccoon_dataset


In [ ]:
def train_model(yaml_path, single=False):
    """Main training function with enhanced configuration"""

    # Load YOLO11 model
    print("🚀 Loading YOLO11n model...")
    model = YOLO("yolo11n.pt")
    
    # Training configuration
    results = model.train(
        data=yaml_path,
        epochs=400,
        imgsz=640,
        batch=16,
        device=0,
        single_cls=single,

        # === AUGMENTATION ===
        hsv_h=0.1, hsv_s=0.9, hsv_v=0.8, 
        degrees=15.0, translate=0.3, scale=0.6, 
        fliplr=0.5, shear=0.2, perspective=0.001, 
        mosaic=1.0, mixup=0.2, copy_paste=0.1,

        # === OPTIMIZATION ===
        lr0=0.001, weight_decay=0.0005, warmup_epochs=3, patience=150, dropout=0.1,

        # === LOSS BALANCE ===
        box=8.0, cls=0.8, dfl=1.5,

        # === TRAINING STABILITY
        cos_lr=True,  
        close_mosaic=30,  
        overlap_mask=False,

        verbose=True,
    )
    
    return model, results

# Train the model

print("🦝 Training raccoon model (single class)...")
raccoon_model, raccoon_results = train_model(yaml_path="raccoon.yaml", single=True)

print("🐭 Training pest model (multi-class)...")
model, results = train_model(yaml_path="pest.yaml")


🦝 Training raccoon model (single class)...
🚀 Loading YOLO11n model...
New https://pypi.org/project/ultralytics/8.3.233 available  Update with 'pip install -U ultralytics'
Ultralytics 8.3.228  Python-3.11.9 torch-2.5.1+cu121 CUDA:0 (NVIDIA GeForce RTX 4070, 12282MiB)
engine\trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=8.0, cache=False, cfg=None, classes=None, close_mosaic=30, cls=0.8, compile=False, conf=None, copy_paste=0.1, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=raccoon.yaml, degrees=15.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.1, dynamic=False, embed=None, epochs=400, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.1, hsv_s=0.9, hsv_v=0.8, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.2, mode=train, model=yolo11m.pt, momentum=0.937, mosaic=1.

In [13]:
def evaluate_model(model, yaml_path):
    """Comprehensive model evaluation"""
    print("📊 Evaluating model...")
    
    # Validation metrics
    metrics = model.val(
        data=yaml_path,
        split="val",
        save_json=True,    # Save results to JSON
        save_hybrid=True,  # Save hybrid version of labels
        conf=0.5,          # Object confidence threshold
        iou=0.6,           # NMS IoU threshold
        plots=True         # Save plots
    )
    
    # Test set evaluation  
    test_metrics = model.val(
        data=yaml_path,
        split="test",
        conf=0.5,
        iou=0.6
    )
    
    return metrics, test_metrics

# Evaluate the trained model

print("🦝 Evaluating the raccoon model (single class)...")
val_metrics_raccoon, test_metrics_raccoon = evaluate_model(raccoon_model, "raccoon.yaml")

print("🐭 Evaluating pest model (multi-class)...")
val_metrics, test_metrics = evaluate_model(model, "pest.yaml")

🦝 Evaluating the raccoon model (single class)...
📊 Evaluating model...
WARNING 'save_hybrid' is deprecated and will be removed in the future.
Ultralytics 8.3.228  Python-3.11.9 torch-2.5.1+cu121 CUDA:0 (NVIDIA GeForce RTX 4070, 12282MiB)
YOLO11m summary (fused): 125 layers, 20,030,803 parameters, 0 gradients, 67.6 GFLOPs
val: Fast image access  (ping: 0.00.0 ms, read: 355.897.2 MB/s, size: 25.3 KB)
val: Scanning C:\Users\plebj\Desktop\School\CS5814\rebalanced_raccoon_dataset\valid\labels.cache... 70 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 70/70 136.4Kit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 5/5 2.4it/s 2.1s0.3ss
                   all         70         76          1          1      0.995      0.809
Speed: 1.5ms preprocess, 10.2ms inference, 0.0ms loss, 0.4ms postprocess per image
Saving C:\Users\plebj\Desktop\School\CS5814\runs\detect\val41\predictions.json...
Results saved to C:\Users\plebj\Deskt

In [6]:
print("\n🎉 Training completed successfully!")
import json

def print_test_metrics(metrics, model_name="Model"):
    """Print test metrics for a given model"""
    print(f"\n📊 {model_name} Test Metrics:")
    
    if metrics is not None:
        # Prefer results_dict for broader version compatibility
        if hasattr(metrics, 'results_dict'):
            precision = metrics.results_dict.get('metrics/precision(B)', None)
            recall = metrics.results_dict.get('metrics/recall(B)', None)
            map50 = metrics.results_dict.get('metrics/mAP50(B)', None)
            map5095 = metrics.results_dict.get('metrics/mAP50-95(B)', None)

            if precision is not None:
                print(f"📊 Test Precision: {precision:.4f}")
            if recall is not None:
                print(f"📊 Test Recall: {recall:.4f}")
            if map50 is not None:
                print(f"📊 Test mAP@0.5: {map50:.4f}")
            if map5095 is not None:
                print(f"📊 Test mAP@0.5:0.95: {map5095:.4f}")
        else:
            print("⚠️ test_metrics does not contain results_dict.")
            
            # Alternative: check for direct attributes (older versions)
            if hasattr(metrics, 'map50'):
                print(f"📊 Test mAP@0.5: {metrics.map50:.4f}")
            if hasattr(metrics, 'map'):
                print(f"📊 Test mAP@0.5:0.95: {metrics.map:.4f}")
            if hasattr(metrics, 'precision'):
                print(f"📊 Test Precision: {metrics.precision:.4f}")
            if hasattr(metrics, 'recall'):
                print(f"📊 Test Recall: {metrics.recall:.4f}")
    else:
        print("📊 Test metrics not available (test_metrics is None)")

# --- Print final metrics for both models ---
print_test_metrics(test_metrics_raccoon, "Raccoon Model")
print_test_metrics(test_metrics, "Pest Model")


🎉 Training completed successfully!

📊 Raccoon Model Test Metrics:
📊 Test Precision: 1.0000
📊 Test Recall: 1.0000
📊 Test mAP@0.5: 0.9950
📊 Test mAP@0.5:0.95: 0.8501

📊 Pest Model Test Metrics:
📊 Test Precision: 0.9500
📊 Test Recall: 0.8816
📊 Test mAP@0.5: 0.9130
📊 Test mAP@0.5:0.95: 0.6800


In [12]:
def load_model(model_path):
    """Load a YOLO model from a .pt file."""
    if not os.path.exists(model_path):
        raise FileNotFoundError(f"Model not found: {model_path}")

    print(f"Loading model: {model_path}")
    model = YOLO(model_path)
    print("Model loaded!\n")
    return model

# Load the model (if needed)
raccoon_model = load_model("runs/detect/train7/weights/best.pt")
model = load_model("runs/detect/train8/weights/best.pt")

Loading model: runs/detect/train7/weights/best.pt
Model loaded!

Loading model: runs/detect/train8/weights/best.pt
Model loaded!



In [ ]:
def export_model(model, export_format="onnnx"):
    """Export model to onnx"""
    print("Exporting model...")

    try:
        path = model.export(format=export_format, opset=14)     
        print(f"✅ Exported to {format}: {path}")
        return {export_format: path}
            
    except Exception as e:
        print(f"❌ Failed to export {format}: {e}")
        return {}

# Export the model
exported_paths = export_model(raccoon_model, "onnx") 
exported_paths = export_model(model, "onnx") 

Loading model: runs/detect/train5/weights/best.pt
Model loaded!

Loading model: runs/detect/train6/weights/best.pt
Model loaded!

Exporting model...
Ultralytics 8.3.228  Python-3.11.9 torch-2.5.1+cu121 CPU (AMD Ryzen 7 9700X 8-Core Processor)
Model summary (fused): 72 layers, 3,005,843 parameters, 0 gradients, 8.1 GFLOPs

PyTorch: starting from 'runs\detect\train5\weights\best.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 5, 8400) (6.0 MB)

ONNX: starting export with onnx 1.19.1 opset 14...
ONNX: slimming with onnxslim 0.1.74...
ONNX: export success  1.4s, saved as 'runs\detect\train5\weights\best.onnx' (11.7 MB)

Export complete (1.6s)
Results saved to C:\Users\plebj\Desktop\School\CS5814\runs\detect\train5\weights
Predict:         yolo predict task=detect model=runs\detect\train5\weights\best.onnx imgsz=640  
Validate:        yolo val task=detect model=runs\detect\train5\weights\best.onnx imgsz=640 data=raccoon.yaml  
Visualize:       https://netron.app
✅ Exporte